In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict

In [2]:
input_file = "global_table_con_UID.xlsx"
output_folder = "training"
os.makedirs(output_folder, exist_ok=True)

In [3]:
#Train01-03 -> 2ESO: 8 grupos de 4 personas y 2 grupos de 3 perosnas (38).
              #2PRI: 8 grupos de 3 personas y 2 grupos de 4 personas (34).
              #5PRI: 8 grupos de 3 personas y 2 grupos de 4 personas (34).
#Train04-08 -> 2ESO: 7 grupos de 4 personas y 3 grupos de 5 perosnas (39).
              #2PRI: 8 grupos de 3 personas y 2 grupos de 4 personas (34).
              #5PRI: 8 grupos de 3 personas y 2 grupos de 4 personas (34).
#Train09-10 -> 2ESO: 7 grupos de 4 personas y 3 grupos de 5 perosnas (39).
              #2PRI: 9 grupos de 3 personas y 1 grupos de 4 personas (35).
              #5PRI: 9 grupos de 3 personas y 1 grupos de 4 personas (35).

# Distribuciones UID por clase y entrenamiento
fold_structure = {
    'Train01': {'2ESO': [4]*8 + [3]*2, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train02': {'2ESO': [4]*8 + [3]*2, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train03': {'2ESO': [4]*8 + [3]*2, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train04': {'2ESO': [4]*7 + [5]*3, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train05': {'2ESO': [4]*7 + [5]*3, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train06': {'2ESO': [4]*7 + [5]*3, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train07': {'2ESO': [4]*7 + [5]*3, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train08': {'2ESO': [4]*7 + [5]*3, '2PRIMARIA': [3]*8 + [4]*2, '5PRIMARIA': [3]*8 + [4]*2},
    'Train09': {'2ESO': [4]*7 + [5]*3, '2PRIMARIA': [3]*9 + [4]*1, '5PRIMARIA': [3]*9 + [4]*1},
    'Train10': {'2ESO': [4]*7 + [5]*3, '2PRIMARIA': [3]*9 + [4]*1, '5PRIMARIA': [3]*9 + [4]*1},
}

In [4]:
xls = pd.ExcelFile(input_file)
df_all = xls.parse(xls.sheet_names[0])
UID_col = "UID"
class_col = "Class"

In [5]:
# Proceso por entrenamiento
for i in range(1, 11):
    test_sheet = f"Test{str(i).zfill(2)}"
    train_name = f"Train{str(i).zfill(2)}"
    
    # Obtener UIDs de test y filtrar entrenamiento
    test_df = xls.parse(test_sheet)
    test_uids = test_df[UID_col].unique()
    train_df = df_all[~df_all[UID_col].isin(test_uids)]
    
    # Obtener UIDs únicos por clase
    uid_class_df = train_df[[UID_col, class_col]].drop_duplicates()
    uids_by_class = {
        cls: uid_class_df[uid_class_df[class_col] == cls][UID_col].tolist()
        for cls in fold_structure[train_name].keys()
    }
    
    # Distribuir UIDs en folds según estructura
    folds_uid = [set() for _ in range(10)]
    for cls, counts in fold_structure[train_name].items():
        np.random.seed(42)
        np.random.shuffle(uids_by_class[cls])
        idx = 0
        for fold_idx, n in enumerate(counts):
            folds_uid[fold_idx].update(uids_by_class[cls][idx:idx + n])
            idx += n

    # Crear archivo Excel
    output_path = os.path.join(output_folder, f"{train_name}.xlsx")
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        # Página "Train"
        train_df.to_excel(writer, index=False, sheet_name="Train")
        # Folds
        for j in range(10):
            fold_df = train_df[train_df[UID_col].isin(folds_uid[j])]
            fold_df.to_excel(writer, index=False, sheet_name=f"Fold{str(j+1).zfill(2)}")